# Combine campaigns into one dataset

Edit `campaigns` and `new_dataset_name`, then run all cells using `backbone_env_v2`.
The notebook combines the raw Parquet files and the existing Step 1 Parquet results,
keeping the pipeline folder structure. Missing raw files are read from archived
Parquet parts. Files are streamed in batches to limit memory use.

All rows are retained, including duplicate posts across campaigns. `source_campaign`
records their origin. Missing columns are filled with nulls; stored pandas indexes
are discarded. Step 1 topic IDs and reuse features remain **campaign-specific**:
use topic IDs together with `source_campaign`. They are not recomputed across campaigns.
Models, embeddings, plots, and checkpoint files are not combined because they belong
to the individual runs. Raw and Step 1 row counts can differ due to preprocessing.
Existing destination files are replaced when rerunning this notebook.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

campaigns = ["Iran_1", "Iran_2", "Iran_3", "Iran_4", "Iran_5", "Iran_6"]
new_dataset_name = "Iran"

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/Labeled_Datasets").is_dir()
)
data_root = project_root / "data/Labeled_Datasets"
results_root = project_root / "results/Labeled_Datasets"
step_1_folder = "step_1_translate_topics_entities"

assert campaigns and len(campaigns) == len(set(campaigns))
assert new_dataset_name not in campaigns
assert all(name not in {".", ".."} and Path(name).name == name
           for name in [*campaigns, new_dataset_name])

raw_output = data_root / new_dataset_name / f"{new_dataset_name}_part_all.gzip.parquet"
step_1_output = (
    results_root / new_dataset_name / f"{new_dataset_name}_part_all"
    / step_1_folder / f"{new_dataset_name}_part_all_step_1.gzip.parquet"
)

In [2]:
def collect_inputs(campaigns, temporary_directory):
    """Find raw and Step 1 files, unpacking archived raw parts when needed.

    Args:
        campaigns: Campaign folder names to combine, in output order.
        temporary_directory: Directory for temporary archived Parquet parts.

    Returns:
        Two lists of (campaign, path) pairs for raw and Step 1 inputs.
    """
    raw_inputs, step_1_inputs = [], []
    for campaign in campaigns:
        folder = data_root / campaign
        raw_path = folder / f"{campaign}_part_all.gzip.parquet"
        step_1_path = (
            results_root / campaign / f"{campaign}_part_all" / step_1_folder
            / f"{campaign}_part_all_step_1.gzip.parquet"
        )
        if not step_1_path.is_file():
            raise FileNotFoundError(step_1_path)
        step_1_inputs.append((campaign, step_1_path))
        if raw_path.is_file():
            raw_inputs.append((campaign, raw_path))
            continue

        parts = sorted(folder.glob(f"{campaign}_part_*.gzip.parquet"))
        if not parts:
            for archive_path in sorted(folder.glob("*.zip")):
                with ZipFile(archive_path) as archive:
                    members = sorted(
                        name for name in archive.namelist()
                        if name.endswith(".parquet") and "part_all" not in Path(name).name
                    )
                    for member in members:
                        part_path = Path(temporary_directory) / f"{len(parts)}_{campaign}.parquet"
                        part_path.write_bytes(archive.read(member))
                        parts.append(part_path)
        if not parts:
            raise FileNotFoundError(f"No raw Parquet files or archived parts for {campaign}")
        raw_inputs.extend((campaign, path) for path in parts)
    return raw_inputs, step_1_inputs


def combine_parquet(inputs, output_path):
    """Concatenate Parquet files with a union schema and campaign provenance.

    Args:
        inputs: Ordered (campaign, path) pairs.
        output_path: Destination gzip Parquet path.

    Returns:
        A DataFrame with source row counts for each campaign.
    """
    schemas = [pq.read_schema(path).remove_metadata() for _, path in inputs]
    schema = pa.unify_schemas(schemas)
    schema = pa.schema([
        field for field in schema
        if not field.name.startswith("__index_level_") and field.name != "source_campaign"
    ]).append(pa.field("source_campaign", pa.string()))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(".tmp")
    counts = []
    with pq.ParquetWriter(temporary_path, schema, compression="gzip") as writer:
        for campaign, path in inputs:
            parquet = pq.ParquetFile(path)
            for batch in parquet.iter_batches(batch_size=25_000):
                table = pa.Table.from_batches([batch])
                arrays = []
                for field in schema:
                    if field.name == "source_campaign":
                        values = pa.array([campaign] * len(table), type=pa.string())
                    elif field.name in table.column_names:
                        values = table[field.name].cast(field.type)
                    else:
                        values = pa.nulls(len(table), type=field.type)
                    arrays.append(values)
                writer.write_table(pa.Table.from_arrays(arrays, schema=schema))
            counts.append({"campaign": campaign, "rows": parquet.metadata.num_rows})
            print(f"{output_path.name}: {campaign}, {parquet.metadata.num_rows:,} rows", flush=True)
    assert pq.read_metadata(temporary_path).num_rows == sum(row["rows"] for row in counts)
    temporary_path.replace(output_path)
    return pd.DataFrame(counts).groupby("campaign", sort=False)[["rows"]].sum()

In [ ]:
with TemporaryDirectory() as temporary_directory:
    raw_inputs, step_1_inputs = collect_inputs(campaigns, temporary_directory)
    raw_counts = combine_parquet(raw_inputs, raw_output)
    step_1_counts = combine_parquet(step_1_inputs, step_1_output)

summary = raw_counts.rename(columns={"rows": "raw_rows"}).join(
    step_1_counts.rename(columns={"rows": "step_1_rows"})
)
summary.loc["Total"] = summary.sum()
summary_directory = project_root / "result" / new_dataset_name
summary_directory.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_directory / "campaign_aggregation_summary.csv")
display(summary.style.format("{:,}"))
print(f"Raw dataset: {raw_output.relative_to(project_root)}")
print(f"Step 1 results: {step_1_output.relative_to(project_root)}")